# Atividade 01 - Filtragem Ótima
**Disciplina:** Sistemas Inteligentes, PPgEE - 2026.1

**Aluno:** Mateus Pincho de Oliveira

---

# Formulação do Filtro Extendido de Kalman (EKF)

## 1. Modelo de Espaço de Estados Discreto e Não-Linear

O Filtro Extendido de Kalman opera sobre um sistema da forma

$$
\mathbf{x}_k = g(\mathbf{x}_{k-1}) + \mathbf{w}_{k-1}, \qquad \mathbf{w}_{k-1} \sim \mathcal{N}(\mathbf{0}, Q)
$$

$$
z_k = h(\mathbf{x}_k) + v_k, \qquad v_k \sim \mathcal{N}(0, R)
$$

em que $g(\cdot)$ é uma **função de transição de estado** e $h(\cdot)$ é **função de medição**. O ruído apresentado no sistema $\{\mathbf{w}_k\}$ e $\{v_k\}$ podem ser descritos como processos de ruído branco gaussiano não-correlacionados.

---

## 2. Linearização via Jacobianos 

Uma vez que $g$ e $h$ são funções não-lineares, o EKF aproxima-as pela expansão em séries de Taylor de primeira ordem avaliada sob o estado atual estimado:

$$
G_k = \left.\frac{\partial g}{\partial \mathbf{x}}\right|_{\mathbf{x} = \hat{\mathbf{x}}_{k-1}}
\qquad \text{(state-transition Jacobian)}
$$

$$
H_k = \left.\frac{\partial h}{\partial \mathbf{x}}\right|_{\mathbf{x} = \hat{\mathbf{x}}_k^-}
\qquad \text{(measurement Jacobian)}
$$

Essas matrizes substituem as matrizes lineares $F$ e $H$ da abordagem original do Filtro de Kalman.

---

## 3. Etapa de Predição

Dada a estimativa a posteriori $\hat{\mathbf{x}}_{k-1}$ e a covariância $P_{k-1}$ do passo anterior:

$$
\hat{\mathbf{x}}_k^- = g(\hat{\mathbf{x}}_{k-1})
$$

$$
P_k^- = G_{k-1}\, P_{k-1}\, G_{k-1}^\top + Q
$$

O subiscrito $^-$ denota uma grandeza **a priori** (predita, pre-medição).

---

## 4. Etapa de Atualização

Dada uma nova medição $z_k$:

**Inovação (resíduo da medição):**
$$
\tilde{z}_k = z_k - h(\hat{\mathbf{x}}_k^-)
$$

**Covariância de Inovação:**
$$
S_k = H_k\, P_k^-\, H_k^\top + R
$$

**Ganho de Kalman:**
$$
K_k = P_k^-\, H_k^\top\, S_k^{-1}
$$

**Estimativa do estado a posteriori:**
$$
\hat{\mathbf{x}}_k = \hat{\mathbf{x}}_k^- + K_k\, \tilde{z}_k
$$

**Covariância a posteriori:**
$$
P_k = (I - K_k H_k)\, P_k^-
$$

---

<!-- ## 5. EKF Algorithm Summary

| Step | Equation | Description |
|------|----------|-------------|
| **Linearize** | $G_{k-1} = \partial g / \partial \mathbf{x}$| ${\hat{\mathbf{x}}_{k-1}}$ | State Jacobian at posterior estimate |
| **Predict state** | $\hat{\mathbf{x}}_k^- = g(\hat{\mathbf{x}}_{k-1})$ | Propagate state through nonlinear model |
| **Predict cov.** | $P_k^- = G_{k-1} P_{k-1} G_{k-1}^\top + Q$ | Propagate covariance (linearized) |
| **Linearize** | $H_k = \partial h / \partial \mathbf{x}$| ${\hat{\mathbf{x}}_k^-}$ | Measurement Jacobian at prior estimate |
| **Innovation** | $\tilde{z}_k = z_k - h(\hat{\mathbf{x}}_k^-)$ | Measurement residual |
| **Gain** | $K_k = P_k^- H_k^\top (H_k P_k^- H_k^\top + R)^{-1}$ | Kalman gain |
| **Update state** | $\hat{\mathbf{x}}_k = \hat{\mathbf{x}}_k^- + K_k \tilde{z}_k$ | Fuse prediction and measurement |
| **Update cov.** | $P_k = (I - K_k H_k) P_k^-$ | Joseph form (first-order) |

> **Note:** The Joseph stabilized form $P_k = (I - K_k H_k) P_k^- (I - K_k H_k)^\top + K_k R K_k^\top$ is numerically safer when the gain is not exactly optimal, but the simpler form above is standard when $K_k$ is computed exactly. -->

In [1]:
import numpy as np

class EKF:
    """
    Generic discrete-time Extended Kalman Filter.

    The filter is model-agnostic: all nonlinear functions and their Jacobians
    are supplied as callables, matching the notation in the formulation above.

    Parameters
    ----------
    g      : callable (n,) -> (n,)   nonlinear state transition  x_k = g(x_{k-1})
    h      : callable (n,) -> (m,)   nonlinear measurement       z_k = h(x_k)
    G_jac  : callable (n,) -> (n,n)  Jacobian dg/dx evaluated at x̂_{k-1}
    H_jac  : callable (n,) -> (m,n)  Jacobian dh/dx evaluated at x̂_k^-
    Q      : (n,n) process noise covariance
    R      : (m,m) measurement noise covariance
    x0     : (n,)  initial state estimate  x̂_0
    P0     : (n,n) initial error covariance P_0
    """

    def __init__(self, g, h, G_jac, H_jac, Q, R, x0, P0):
        self.g = g
        self.h = h
        self.G_jac = G_jac
        self.H_jac = H_jac
        self.Q = np.asarray(Q, dtype=float)
        self.R = np.asarray(R, dtype=float)
        self.x = np.asarray(x0, dtype=float).copy()
        self.P = np.asarray(P0, dtype=float).copy()
        self.n = len(self.x)

    def predict(self):
        """Prediction step: propagate state and covariance."""
        G = self.G_jac(self.x)           # linearize at x̂_{k-1}
        self.x = self.g(self.x)          # x̂_k^- = g(x̂_{k-1})
        self.P = G @ self.P @ G.T + self.Q  # P_k^- = G P G^T + Q

    def update(self, z):
        """Update step: incorporate measurement z."""
        z = np.asarray(z, dtype=float)
        H = self.H_jac(self.x)                        # linearize at x̂_k^-
        S = H @ self.P @ H.T + self.R                  # innovation covariance
        K = self.P @ H.T @ np.linalg.inv(S)            # Kalman gain
        innovation = z - self.h(self.x)                # z_k - h(x̂_k^-)
        self.x = self.x + K @ innovation               # x̂_k = x̂_k^- + K * innovation
        I = np.eye(self.n)
        self.P = (I - K @ H) @ self.P                  # P_k = (I - K H) P_k^-

    def step(self, z):
        """
        One full predict + update cycle.

        Parameters
        ----------
        z : array-like, shape (m,) or scalar
            Measurement at current time step.

        Returns
        -------
        x : (n,) posterior state estimate x̂_k
        P : (n,n) posterior error covariance P_k
        """
        self.predict()
        self.update(z)
        return self.x.copy(), self.P.copy()

Implement a navigation system model and use Extend Kalman Filter

- The object moves in X and Y direction accordly to the position-velocity model
- Ts = 1 second
- The system measures the distance $d_i$ from the object to the base_station at points with coordinates $X_i$ and $Y_i$. There are four base stations

PVA model 

- accelaration is constant
- $v = v_0 + at$
- $p = p_0 + v_0t + \frac{at^2}{2}$


x
vx
ax
y
vy
ay






In [ ]:
import numpy as np

class EKF:
    """
    Generic discrete-time Extended Kalman Filter.

    The filter is model-agnostic: all nonlinear functions and their Jacobians
    are supplied as callables, matching the notation in the formulation above.

    Parameters
    ----------
    g      : callable (n,) -> (n,)   nonlinear state transition  x_k = g(x_{k-1})
    h      : callable (n,) -> (m,)   nonlinear measurement       z_k = h(x_k)
    G_jac  : callable (n,) -> (n,n)  Jacobian dg/dx evaluated at x̂_{k-1}
    H_jac  : callable (n,) -> (m,n)  Jacobian dh/dx evaluated at x̂_k^-
    Q      : (n,n) process noise covariance
    R      : (m,m) measurement noise covariance
    x0     : (n,)  initial state estimate  x̂_0
    P0     : (n,n) initial error covariance P_0
    """

    def __init__(self, G, h, H_jac, Q, R, x0, P0):

        self.G = G
        self.h = h
        self.H_jac = H_jac
        self.Q = np.asarray(Q, dtype=float)
        self.w = None # noise param
        self.R = np.asarray(R, dtype=float)
        self.x = np.asarray(x0, dtype=float).copy()
        self.P = np.asarray(P0, dtype=float).copy()
        self.n = len(self.x)

    def predict(self):
        """Prediction step: propagate state and covariance."""
        self.x = self.G @ self.x + self.w             # x̂_k^- = G @(x̂_{k-1}) + w_k
        self.P = self.G @ self.P @ self.G.T + self.Q  # P_k^- = G P G^T + Q

    def update(self, z):
        """Update step: incorporate measurement z."""
        z = np.asarray(z, dtype=float)
        H = self.H_jac(self.x)                        # linearize at x̂_k^-
        S = H @ self.P @ H.T + self.R                  # innovation covariance
        K = self.P @ H.T @ np.linalg.inv(S)            # Kalman gain
        innovation = z - self.h(self.x)                # z_k - h(x̂_k^-)
        self.x = self.x + K @ innovation               # x̂_k = x̂_k^- + K * innovation
        I = np.eye(self.n)
        self.P = (I - K @ H) @ self.P                  # P_k = (I - K H) P_k^-

    def step(self, z):
        """
        One full predict + update cycle.

        Parameters
        ----------
        z : array-like, shape (m,) or scalar
            Measurement at current time step.

        Returns
        -------
        x : (n,) posterior state estimate x̂_k
        P : (n,n) posterior error covariance P_k
        """
        self.predict()
        self.update(z)
        return self.x.copy(), self.P.copy()

In [2]:
F = np.array([[1, Ts, (Ts**2)/2, 0, 0, 0],
              [ 0, 1, Ts, 0, 0, 0],
              [ 0, 0, 1,  0, 0, 0],
              [ 0, 0, 0, 1, Ts, (Ts**2)/2],
              [ 0, 0, 0, 0, 1, Ts],
              [ 0, 0, 0, 0, 0, 1]])

NameError: name 'Ts' is not defined